In [1]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

In [2]:
# # test sim_bcs
# from superconductivity.utilities.constants import G0_muS
# from superconductivity.utilities.meta.axis import axis
# from superconductivity.utilities.meta.param import param
# from superconductivity.models.bcs import sim_bcs

# V_mV = axis("V_mV", -1.0, 1.0, 101)

# GN_G0 = axis("GN_G0", 0, 1, 101)
# T_K = axis("T_K", 0, 1.3, 14)
# Delta_meV = axis("Delta_meV", 0.18, 0.2, 21)
# gamma_meV = axis("gamma_meV", 0.0, 4.0, 11)
# A_mV = axis("A_mV", 0.1, 1.0, 10)
# nu_GHz = axis("nu_GHz", 1, 10, 10)
# sigmaV_mV = axis("sigmaV_mV", 1, 10, 10)

# GN_G0 = param("GN_G0", 0.2)
# T_K = param("T_K", 0.236)
# Delta_meV = param("Delta_meV", 0.18)
# gamma_meV = param("gamma_meV", 0.0)
# # A_mV = param("A_mV", 0)
# nu_GHz = param("nu_GHz", 10.0)
# # sigmaV_mV = param("sigmaV_mV", 1.0)

# bcs_v = sim_bcs(
#     V_mV=V_mV,
#     GN_G0=GN_G0,
#     T_K=T_K,
#     Delta_meV=Delta_meV,
#     gamma_meV=gamma_meV,
#     A_mV=A_mV,
#     nu_GHz=nu_GHz,
#     sigmaV_mV=sigmaV_mV,
# )

In [ ]:
# load cache
from superconductivity.utilities import make_cache, load_cache, save_cache

cache = make_cache(name="TB-18.3GHz")
save_cache(cache)

cache = load_cache("TB-18.3GHz")

AttributeError: Can't get attribute 'linear' on <module '__main__'>

In [ ]:
# eva gui
from superconductivity.deprecated_gui import gui

importlib.reload(sys.modules["superconductivity.deprecated_gui"])

app = gui()

In [ ]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_amplitudes_18.3000GHz",
)
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
save_cache(cache)

In [ ]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="GHz_",
    strip1="V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
        ("no_irradiation", 0.005),
    ],
    limits=(None, None),
    norm=1e-3,
    label="Aout_mV",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
save_cache(cache)

In [ ]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
save_cache(cache)

In [11]:
# offset analysis
from superconductivity.evaluation import OffsetSpec
from superconductivity.evaluation import offset_analysis

cache.offsetspec = OffsetSpec(
    Vbins_mV=np.linspace(-0.5, 0.5, 51),
    Ibins_nA=np.linspace(-5.0, 5.0, 51),
    Voffscan_mV=np.linspace(-0.045, 0.045, 451),
    Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
    cutoff_Hz=43.7,
    sampling_Hz=137.0,
)

cache.offsetanalysis = offset_analysis(
    traces=cache.traces,
    spec=cache.offsetspec,
)
save_cache(cache)

offset_analysis:   0%|          | 0/143 [00:00<?, ?trace/s]

In [13]:
# sampling

from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 1601),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=43.7,
    sampling_Hz=137.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
save_cache(cache)

sampling:   0%|          | 0/143 [00:00<?, ?trace/s]

PosixPath('/Users/oliver/Documents/cryolab/superconductivity/.cache/TB-18.3GHz.pkl')

In [ ]:
# reduced units

from superconductivity.utilities import reduced_units

cache.exp_v, cache.exp_i = reduced_units(
    (cache.exp_v, cache.exp_i),
    Delta_meV=0.195,
    GN_G0=0.189,
    nu_GHz=18.3,
)
save_cache(cache)

In [ ]:
# calibrate

from superconductivity.evaluation import CalibrationSpec
from superconductivity.evaluation import calibrate

cache.calibrationspec = CalibrationSpec(
    label="A_mV",
    lookup=0.002173 * cache.exp_v.yaxis,
)
cache.cal_v, cache.cal_i = calibrate(
    cache.exp_v,
    cache.exp_i,
    calibrationspec=cache.calibrationspec,
)
save_cache(cache)

PosixPath('/Users/oliver/Documents/cryolab/superconductivity/.cache/TB-18.3GHz.pkl')

In [16]:
# mapping
from superconductivity.utilities import mapping

cache.map_v, cache.map_i = mapping(
    (cache.cal_v, cache.cal_i),
    axis="A_mV",
    xbins=np.linspace(0, 1.5, 151),
    fill="interpolate",
)
save_cache(cache)

PosixPath('/Users/oliver/Documents/cryolab/superconductivity/.cache/TB-18.3GHz.pkl')